In [1]:
"""
Step 5 of 5 — verification.

Reads the result files produced by steps 01-04 and checks every number quoted
in the paper against them. Each line prints PASS or FAIL with the computed
value beside the manuscript value, and the last line is a count. Nothing is
recomputed from scratch except where the paper's claim IS a derivation (the
n=3 identity, the generalised-mean comparison, the Irish zero crossing).

Run it after 01-04. If a number in the manuscript is ever edited without the
pipeline being rerun, this notebook fails loudly.

Expected result: 86 of 86 checks passed.
"""
import os, glob, itertools
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ setup ---
TARGET = 'OWAGDP Data and Code'
BASE = f'/content/drive/MyDrive/{TARGET}'

if not os.path.isdir(BASE):
    try:
        from google.colab import drive
        if not os.path.isdir('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        BASE = next(dp for dp, dn, fn in os.walk('/content/drive/MyDrive')
                    if os.path.basename(dp) == TARGET)
    except Exception:
        BASE = os.getcwd()          # running from a clone of the repository
print('BASE =', BASE, '\n')


def find(name):
    hits = sorted(glob.glob(os.path.join(BASE, '**', name), recursive=True))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {BASE}')
    return hits[0]


INST = ['IMF', 'OECD', 'EC']
_res = []


def chk(label, got, want, tol=0.0):
    """tol=0 means exact equality (ints, strings, booleans)."""
    if isinstance(want, str):
        ok = (str(got) == want)
        shown = f'{got}'
    else:
        ok = abs(float(got) - float(want)) <= tol
        shown = f'{float(got):.4f}'
    _res.append((ok, label))
    print(f'  [{"PASS" if ok else "FAIL"}]  {label:<58} {shown:>10}   paper {want}')


def me_weights(alpha):
    """Maximum-entropy OWA weights, n=3, w[0] applied to the LARGEST value."""
    if abs(alpha - 0.5) < 1e-12:
        return np.full(3, 1/3)
    h = (2 - 4*alpha + np.sqrt(4 + 48*alpha - 48*alpha**2)) / (8*alpha)
    w = np.array([1.0, h, h**2]); w /= w.sum()
    assert abs((2*w[0] + w[1])/2 - alpha) < 1e-12
    return w


def gowa(w, g, lam):
    """Generalised OWA. At lambda = 0 the limit is the geometric mean."""
    if abs(lam) < 1e-9:
        return float(np.prod(g ** w))
    return float((w @ (g ** lam)) ** (1 / lam))


agg = pd.read_csv(find('aggregates.csv'))
print('aggregates          ', find('aggregates.csv').replace(BASE, '.'))
err = pd.read_csv(find('forecast_errors.csv'))
print('forecasts+outcomes  ', find('forecast_errors.csv').replace(BASE, '.'))
wts = pd.read_csv(find('weights.csv'))
mae = pd.read_csv(find('mae.csv'))
ind = pd.read_csv(find('induced.csv'))
rsc = pd.read_csv(find('rescore.csv'))
unf = pd.read_csv(find('uncertainty_full.csv'))
une = pd.read_csv(find('uncertainty_excl2021.csv'))

# ------------------------------------------- Table I and Section III-B -------
print('\n=== Table I: maximum-entropy weights (n = 3) ===')
bad = 0
for _, r in wts.iterrows():
    w = me_weights(float(r.alpha))
    if max(abs(w - np.array([r.w1, r.w2, r.w3]))) > 5e-4:
        bad += 1
chk('all 9 stored weight vectors reproduce', bad, 0)
chk('columns sum to one within 3-decimal rounding',
    abs(wts[['w1', 'w2', 'w3']].sum(axis=1) - 1).max(), 0.0, 2e-3)
chk('w1 - w3 = 2a - 1 (max deviation)',
    (wts.w1 - wts.w3 - (2*wts.alpha - 1)).abs().max(), 0.0, 1e-3)

# ------------------------------------------------------ Section III-C --------
print('\n=== Section III-C: the n = 3 identity ===')
chk('spread == owa_0.8 - owa_0.2 (max deviation)',
    (agg.spread - (agg['owa_0.8'] - agg['owa_0.2'])).abs().max(), 0.0, 5e-4)
chk('spread == 0.6 x range        (max deviation)',
    (agg.spread - 0.6*agg['range']).abs().max(), 0.0, 5e-4)

# --------------------------------------------- Section V-A and Table II ------
print('\n=== Section V-A: panel and Table II ===')
chk('aggregation cells', len(agg), 74)
chk('economies', agg.iso.nunique(), 37)
chk('vintage rows (36 x 5 + Croatia)', len(err), 181)
comp = err.groupby('iso').size()
chk('economies with a complete record', (comp == 5).sum(), 36)
chk('economies dropped from the induced analysis', (comp < 5).sum(), 1)

rnds = sorted(err['round'].unique())
chk('scoring rounds', len(rnds), 5)
chk('autumn rounds (14-month horizon)', sum(x.startswith('autumn') for x in rnds), 3)
chk('spring rounds (20-month horizon)', sum(x.startswith('spring') for x in rnds), 2)
chk('each round scores exactly one target year',
    int(err.groupby('round').target.nunique().max()), 1)

w2_, w8_ = me_weights(0.2), me_weights(0.8)
worst = 0.0
for _, r in agg[agg.iso.isin(['SWE', 'DNK', 'ITA', 'IRL'])].iterrows():
    g = np.sort(r[INST].to_numpy(float))[::-1]
    worst = max(worst, abs(w2_ @ g - r['owa_0.2']), abs(w8_ @ g - r['owa_0.8']))
chk('Table II rows reproduce from the weights', worst, 0.0, 5e-4)

# ------------------------------------------------------- Section V-B ---------
print('\n=== Section V-B: roles and the attitudinal spread ===')
chk('IMF most optimistic',  (agg.most_opt == 'IMF').sum(), 34)
chk('IMF most pessimistic', (agg.most_pes == 'IMF').sum(), 15)
chk('EC most pessimistic',  (agg.most_pes == 'EC').sum(), 30)
chk('EC most optimistic',   (agg.most_opt == 'EC').sum(), 23)
piv = agg.pivot(index='iso', columns='year', values=['most_opt', 'most_pes'])
chk('economies where a role changes between years',
    ((piv['most_opt'][2026] != piv['most_opt'][2027]) |
     (piv['most_pes'][2026] != piv['most_pes'][2027])).sum(), 27)
chk('median spread', round(agg.spread.median(), 2), 0.16, 1e-9)
chk('mean spread',   round(agg.spread.mean(), 2),   0.23, 1e-9)
chk('near-consensus cells below 0.05', (agg.spread < 0.05).sum(), 5)
ita = agg[(agg.iso == 'ITA') & (agg.year == 2026)].iloc[0]
chk('Italy 2026 spread', round(ita.spread, 3), 0.015, 1e-9)
irl = agg[(agg.iso == 'IRL') & (agg.year == 2026)].iloc[0]
chk('maximum spread (Ireland 2026)', round(irl.spread, 2), 2.24, 1e-9)
chk('Ireland 2026 IMF forecast',  round(irl.IMF, 2),   2.52, 1e-9)
chk('Ireland 2026 OECD forecast', round(irl.OECD, 2), -1.04, 1e-9)
chk('Ireland 2026 EC forecast',   round(irl.EC, 2),   -1.22, 1e-9)
chk('spread interquartile range, lower', round(agg.spread.quantile(.25), 2), 0.10, 1e-9)
chk('spread interquartile range, upper', round(agg.spread.quantile(.75), 2), 0.24, 1e-9)

g_irl = np.sort(irl[INST].to_numpy(float))[::-1]
lo, hi = 0.05, 0.95
for _ in range(200):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if me_weights(mid) @ g_irl < 0 else (lo, mid)
chk('Ireland 2026 crosses zero at alpha', round((lo + hi)/2, 3), 0.477, 1e-9)

dnk = np.sort(agg[(agg.iso == 'DNK') & (agg.year == 2026)].iloc[0][INST].to_numpy(float))[::-1]
sh = [gowa(w8_, dnk, l) - gowa(w2_, dnk, l) for l in np.linspace(-1, 2, 301)]
chk('Denmark 2026: alpha shift, minimum over lambda', round(min(sh), 2), 0.36, 1e-9)
chk('Denmark 2026: alpha shift, maximum over lambda', round(max(sh), 2), 0.39, 1e-9)
eq = np.full(3, 1/3)
chk('Denmark 2026: lambda shift at alpha = 0.5',
    round(gowa(eq, dnk, 2) - gowa(eq, dnk, -1), 2), 0.05, 1e-9)

# ------------------------------------------------------- Section V-C ---------
print('\n=== Section V-C: accuracy and the induced operator ===')
chk('IMF panel MAE',  round(mae.IMF.mean(), 2),  1.80, 1e-9)
chk('OECD panel MAE', round(mae.OECD.mean(), 2), 1.91, 1e-9)
chk('EC panel MAE',   round(mae.EC.mean(), 2),   1.84, 1e-9)
chk('IMF most accurate in',  (mae.best == 'IMF').sum(),  16)
chk('OECD most accurate in', (mae.best == 'OECD').sum(), 10)
chk('EC most accurate in',   (mae.best == 'EC').sum(),   10)
chk('induced cells', len(ind), 72)
chk('cells changed by the accuracy ordering', (ind.diff_acc.abs() > 1e-9).sum(), 63)
chk('Proposition 3 violations (diff_acc > 0)', (ind.diff_acc > 1e-9).sum(), 0)
chk('mean |change| over all 72 cells', round(ind.diff_acc.abs().mean(), 2), 0.12, 1e-9)
chk('mean |change| over the 63 that change',
    round(ind.loc[ind.diff_acc.abs() > 1e-9, 'diff_acc'].abs().mean(), 2), 0.14, 1e-9)
chk('cells changed by the recency ordering', (ind.diff_rec.abs() > 1e-9).sum(), 67)
chk('mean |change| under recency', round(ind.diff_rec.abs().mean(), 2), 0.15, 1e-9)
i26 = ind[(ind.iso == 'IRL') & (ind.year == 2026)].iloc[0]
chk('Ireland 2026 magnitude-ordered', round(i26.owa, 2), 1.37, 1e-9)
chk('Ireland 2026 accuracy-ordered',  round(i26.iowa_acc, 2), -0.87, 1e-9)
chk('Ireland 2026 accuracy order', i26.order_acc, 'EC-OECD-IMF')
chk('Ireland 2027 orderings coincide',
    abs(ind[(ind.iso == 'IRL') & (ind.year == 2027)].iloc[0].diff_acc), 0.0, 1e-9)
oth = ind[ind.iso != 'IRL']
chk('next largest divergence (Mexico 2026)', round(oth.diff_acc.min(), 2), -0.40, 1e-9)
chk('largest divergence outside Ireland and Mexico',
    round(oth[oth.iso != 'MEX'].diff_acc.min(), 2), -0.35, 1e-9)
chk('cells where the sign of the aggregate flips',
    ((np.sign(ind.owa) != np.sign(ind.iowa_acc)) & (ind.diff_acc.abs() > 1e-9)).sum(), 1)
chk('rescoring changes the most accurate institution in', int(rsc.changed.sum()), 1)

# ------------------------------------------------------- Section V-D ---------
print('\n=== Section V-D: forecast uncertainty ===')
for lab, d, N, med, lo_i, lo_v, hi_i, hi_v, xirl in [
        ('full',   unf, 125, 3.66, 'BEL', 1.16, 'IRL', 16.54, 3.64),
        ('ex2021', une,  64, 1.96, 'TUR', 0.34, 'IRL',  9.55, 1.83)]:
    chk(f'{lab}: support size', d.n_support.iloc[0], N)
    p = d.pivot(index='iso', columns='year', values='width_0.5')
    chk(f'{lab}: alpha=0.5 width identical across years',
        (p[2026] - p[2027]).abs().max(), 0.0, 1e-9)
    chk(f'{lab}: median width', round(d['width_0.5'].median(), 2), med, 1e-9)
    u = d.drop_duplicates('iso').sort_values('width_0.5')
    chk(f'{lab}: narrowest economy', u.iloc[0].iso, lo_i)
    chk(f'{lab}: narrowest width',   round(u.iloc[0]['width_0.5'], 2), lo_v, 1e-9)
    chk(f'{lab}: widest economy',    u.iloc[-1].iso, hi_i)
    chk(f'{lab}: widest width',      round(u.iloc[-1]['width_0.5'], 2), hi_v, 1e-9)
    chk(f'{lab}: median width excluding Ireland',
        round(d.loc[d.iso != 'IRL', 'width_0.5'].median(), 2), xirl, 1e-9)

m = unf.merge(une, on=['iso', 'year'], suffixes=('_f', '_e'))
shift = m['med_0.5_e'] - m['med_0.5_f']
chk('excluding 2021 shifts ranges down in N economies',
    (shift.groupby(m.iso).mean() < 0).sum(), 34)
chk('median downward shift', round(shift.median(), 2), -0.50, 1e-9)

ms = agg[agg.iso.isin(set(unf.iso))].spread.median()
r1 = unf['width_0.5'].median() / ms
r2 = une['width_0.5'].median() / ms
chk('full window / median spread  (paper: more than 20)', r1 > 20, True)
chk('excl 2021 / median spread    (paper: more than 12)', r2 > 12, True)
print(f'         measured ratios: {r1:.1f}x and {r2:.1f}x')

for k in INST:
    err['s_' + k] = err[k] - err['actual']
keep = sorted(comp[comp == 5].index)
wr = []
for iso in keep:
    h = err[err.iso == iso]
    combos = h[['s_' + k for k in INST]].to_numpy(float)
    for _, row in agg[agg.iso == iso].iterrows():
        v = np.sort(row[INST].to_numpy(float) - combos, axis=1)[:, ::-1] @ np.full(3, 1/3)
        q = np.percentile(v, [2.5, 97.5], method='inverted_cdf')
        wr.append(q[1] - q[0])
chk('whole-round median width', round(float(np.median(wr)), 2), 4.25, 1e-9)

# ------------------------------------------------------- Section V-E ---------
print('\n=== Section V-E: limitations ===')
k5 = err.iso.isin(keep)
a21 = err[k5 & (err.target == 2021)]
a22 = err[k5 & (err.target != 2021)]
chk('MAE for the 2021 target',
    round(a21[INST].sub(a21['actual'], axis=0).abs().to_numpy().mean(), 2), 2.95, 1e-9)
chk('MAE for 2022-2023',
    round(a22[INST].sub(a22['actual'], axis=0).abs().to_numpy().mean(), 2), 1.58, 1e-9)
s21 = err[err.target == 2021]
chk('2021 signed error, IMF',  round((s21.IMF  - s21.actual).mean(), 2), -2.02, 1e-9)
chk('2021 signed error, OECD', round((s21.OECD - s21.actual).mean(), 2), -3.78, 1e-9)
chk('2021 signed error, EC',   round((s21.EC   - s21.actual).mean(), 2), -2.75, 1e-9)
chk('economies with adjacent MAE gap under 2% of the pair mean',
    (mae.min_gap_pct < 2).sum(), 11)
chk('Canada adjacent MAE gap',
    round(float(mae.loc[mae.iso == 'CAN', 'min_gap'].iloc[0]), 3), 0.001, 1e-9)
chk('Ireland adjacent MAE gap',
    round(float(mae.loc[mae.iso == 'IRL', 'min_gap'].iloc[0]), 3), 0.003, 1e-9)
chk('Ireland 2026 swap moves the aggregate by', round(i26.swap_effect, 2), 0.55, 1e-9)
chk('Ireland 2026 after the swap (still negative)',
    round(i26.iowa_acc_swap23, 2), -0.32, 1e-9)

# --------------------------------------------------------------- summary -----
n_ok = sum(1 for ok, _ in _res if ok)
print(f'\n{"="*72}\n{n_ok} of {len(_res)} checks passed')
for ok, lab in _res:
    if not ok:
        print('  FAILED:', lab)


Mounted at /content/drive
BASE = /content/drive/MyDrive/OWAGDP Data and Code 

aggregates           ./02_aggregate/Results/aggregates.csv
forecasts+outcomes   ./03a_build_error/Result/forecast_errors.csv

=== Table I: maximum-entropy weights (n = 3) ===
  [PASS]  all 9 stored weight vectors reproduce                          0.0000   paper 0
  [PASS]  columns sum to one within 3-decimal rounding                   0.0010   paper 0.0
  [PASS]  w1 - w3 = 2a - 1 (max deviation)                               0.0000   paper 0.0

=== Section III-C: the n = 3 identity ===
  [PASS]  spread == owa_0.8 - owa_0.2 (max deviation)                    0.0001   paper 0.0
  [PASS]  spread == 0.6 x range        (max deviation)                   0.0001   paper 0.0

=== Section V-A: panel and Table II ===
  [PASS]  aggregation cells                                             74.0000   paper 74
  [PASS]  economies                                                     37.0000   paper 37
  [PASS]  vintage rows